In [1]:
import os
from pathlib import Path
import pandas as pd
import gc


BASE_PATH = Path.cwd().parents[0]
RAW_PATH = BASE_PATH / "data" / "raw"
BRONZE_PATH = BASE_PATH / "data" / "bronze"

BRONZE_PATH.mkdir(parents=True, exist_ok=True)

In [2]:
ais_files = sorted(RAW_PATH.glob("AIS_*.csv"))
print(f"Total AIS files found: {len(ais_files)}")

Total AIS files found: 92


In [3]:
for file in ais_files:
    print(f"\nProcessing: {file.name}")

    # Read one day only
    df = pd.read_csv(file)

    # ---- Cargo vessels (70–79) ----
    cargo_df = df[df["VesselType"].between(70, 79)]

    # ---- LA PORT (BERTH) ----
    la_port_df = cargo_df[
        (cargo_df["SOG"] == 0) &
        (cargo_df["LAT"].between(33.70, 33.80)) &
        (cargo_df["LON"].between(-118.30, -118.20))
    ].copy()
    la_port_df["location_type"] = "port"

    # ---- LA ANCHORAGE ----
    la_anchorage_df = cargo_df[
        (cargo_df["SOG"] > 0) &
        (cargo_df["SOG"] <= 0.5) &
        (cargo_df["LAT"].between(33.55, 33.75)) &
        (cargo_df["LON"].between(-118.45, -118.15))
    ].copy()
    la_anchorage_df["location_type"] = "anchorage"

    # ---- Combine ----
    la_combined_df = pd.concat(
        [la_port_df, la_anchorage_df],
        ignore_index=True
    )

    # ---- SAVE AS CSV (IMPORTANT CHANGE) ----
    out_file = BRONZE_PATH / f"la_port_anchorage_{file.stem}.csv"
    la_combined_df.to_csv(out_file, index=False)

    print(f"Saved: {out_file.name} | rows: {la_combined_df.shape[0]}")

    # ---- Free memory ----
    del df, cargo_df, la_port_df, la_anchorage_df, la_combined_df
    gc.collect()



Processing: AIS_2024_05_01.csv
Saved: la_port_anchorage_AIS_2024_05_01.csv | rows: 9059

Processing: AIS_2024_05_02.csv
Saved: la_port_anchorage_AIS_2024_05_02.csv | rows: 9808

Processing: AIS_2024_05_03.csv
Saved: la_port_anchorage_AIS_2024_05_03.csv | rows: 11872

Processing: AIS_2024_05_04.csv
Saved: la_port_anchorage_AIS_2024_05_04.csv | rows: 10153

Processing: AIS_2024_05_05.csv
Saved: la_port_anchorage_AIS_2024_05_05.csv | rows: 9201

Processing: AIS_2024_05_06.csv
Saved: la_port_anchorage_AIS_2024_05_06.csv | rows: 9327

Processing: AIS_2024_05_07.csv
Saved: la_port_anchorage_AIS_2024_05_07.csv | rows: 9305

Processing: AIS_2024_05_08.csv
Saved: la_port_anchorage_AIS_2024_05_08.csv | rows: 8523

Processing: AIS_2024_05_09.csv
Saved: la_port_anchorage_AIS_2024_05_09.csv | rows: 9288

Processing: AIS_2024_05_10.csv
Saved: la_port_anchorage_AIS_2024_05_10.csv | rows: 10862

Processing: AIS_2024_05_11.csv
Saved: la_port_anchorage_AIS_2024_05_11.csv | rows: 9554

Processing: AIS_2

In [5]:
files = list(BRONZE_PATH.glob("la_port_anchorage_*.csv"))

final_la_df = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)

print("Final LA dataset shape:", final_la_df.shape)

Final LA dataset shape: (1019607, 18)


In [7]:
out_file = BRONZE_PATH / f"final_la_df.csv"
final_la_df.to_csv(out_file, index=False)